<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH09/CH09_NB02_MOE_Upcycling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models

### Chapter 9: Mixture of Experts (MoE)
### Notebook 2: Upcycling a pre-trained expert

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU T4

Models:
- HuggingFaceTB/SmolLM2-1.7B-Instruct
- oopere/SmolLM2-1.7B-ClinicalNER

_____
In Notebook 1, you trained both an MoE expert and a router. In this notebook, you reuse the clinical MLP learned in Chapter 7 and transplant it directly as Expert 1.

This upcycling workflow skips full expert training and focuses on router adaptation, then evaluates general behavior, clinical extraction quality, and routing decisions.

# Setting up the notebook

This section installs dependencies, imports modules, and defines utility and configuration values used throughout the notebook.

In [1]:
!pip install -q transformers datasets torch accelerate

In [2]:
import copy, random
import gc
import torch, json
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [3]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✓ Random seed set to 42")

✓ Random seed set to 42


## Configuration

Centralized parameters for reproducible experiments. Adjust values here before running the workflow.

In [ ]:
# Core model configuration
BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
CLINICAL_MODEL_ID = "oopere/SmolLM2-1.7B-ClinicalNER"

# Runtime and training settings
EPOCHS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Set deterministic seeds used in this notebook
set_seed(SEED)
torch.manual_seed(0)
random.seed(0)

print(f"Configuration loaded | device={DEVICE} | epochs={EPOCHS} | seed={SEED}")

Running on: cuda


In [5]:
def clear_gpu_cache():
    """Clear GPU cache completely"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

## Workflow: Building the upcycled MoE

### Loading the two models

We need both models in memory only for a short phase. After extracting the clinical MLP weights from the Chapter 7 model and transplanting them into the MoE blocks, we release that model before router training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# The base model becomes Expert 0 (general knowledge).
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID,
                                             torch_dtype=torch.float16)
model.to(DEVICE)

print(f"\nBase model     : {sum(p.numel() for p in model.parameters()):,} parameters")

## The MoE block

The class is identical to Notebook 1 with one addition: an optional
`clinical_mlp` argument. When provided, Expert 1 is initialised directly
from those weights instead of from a copy of the base MLP.

That single optional argument is the entire difference between training an
expert from scratch and reusing one you already have.

In [7]:
class Trainable2ExpertMoE(nn.Module):
    """Two-expert soft MoE block.

    Args:
        base_mlp:     the original MLP from the decoder layer (becomes Expert 0).
        hidden_size:  model hidden dimension, used to size the router.
        clinical_mlp: pre-trained MLP to use as Expert 1. When provided, no
                      training of Expert 1 is needed — the specialisation is
                      already in the weights.
    """
    def __init__(self, base_mlp, hidden_size, clinical_mlp=None):
        super().__init__()
        target_dtype  = next(base_mlp.parameters()).dtype
        target_device = next(base_mlp.parameters()).device

        self.expert_general  = base_mlp
        self.expert_clinical = copy.deepcopy(
              clinical_mlp if clinical_mlp is not None else base_mlp
          ).to(device=target_device, dtype=target_dtype)
        self.router = nn.Linear(hidden_size, 2, bias=False,
                                dtype=target_dtype, device=target_device)
        self.last_routing = None

    def forward(self, hidden_states):
        router_logits = self.router(hidden_states)
        routing_weights = F.softmax(router_logits.float(), dim=-1).to(hidden_states.dtype)

        out_general  = self.expert_general(hidden_states)
        out_clinical = self.expert_clinical(hidden_states)

        output = out_general  * routing_weights[..., 0].unsqueeze(-1) + \
                out_clinical * routing_weights[..., 1].unsqueeze(-1)

        self.last_routing = routing_weights.detach()
        return output



## Transplanting the pre-trained expert

For each target layer we extract the MLP from the chapter-7 model and pass
it as `clinical_mlp`. The base model keeps its original MLP as Expert 0.
After the transplant we delete the clinical model to free GPU memory.

In [8]:
LAYERS_TO_MODIFY = range(model.config.num_hidden_layers)


In [ ]:
# Extract only the MLP weights from the chapter-7 model — CPU only.
hidden_size = model.config.hidden_size
print("Extracting clinical MLPs from chapter-7 model...")
clinical_mlps = {}
clinical_model = AutoModelForCausalLM.from_pretrained(
    CLINICAL_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu"
)
for layer_idx in LAYERS_TO_MODIFY:
    clinical_mlps[layer_idx] = copy.deepcopy(
        clinical_model.model.layers[layer_idx].mlp
    )

# Chapter-7 model has served its purpose — free the memory before surgery.
del clinical_model
torch.cuda.empty_cache()
print("Clinical model unloaded.")

# Now perform the surgery with only one model in GPU memory.
for layer_idx in LAYERS_TO_MODIFY:
    base_mlp = model.model.layers[layer_idx].mlp
    model.model.layers[layer_idx].mlp = Trainable2ExpertMoE(
        base_mlp, hidden_size, clinical_mlp=clinical_mlps[layer_idx]
    )

del clinical_mlps
torch.cuda.empty_cache()
print(f"Transplant complete: {len(LAYERS_TO_MODIFY)} layers using chapter-7 Expert 1.")

In [10]:
gc.collect()

260

## Freezing strategy

In Notebook 1 you unfroze Expert 1 AND the router. Here Expert 1 is already
trained — freezing it is not only possible but correct. The only parameters
that need to learn are the router weights, one linear layer per MoE block.

In [11]:
# Freeze the entire model.
for param in model.parameters():
    param.requires_grad_(False)

# Unfreeze only the routers — Expert 1 stays frozen.
for layer_idx in LAYERS_TO_MODIFY:
    for param in model.model.layers[layer_idx].mlp.router.parameters():
        param.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters  : {total:,}")
print(f"Trainable (router): {trainable:,}  ({trainable/total:.4%})")
print()

Total parameters  : 2,919,434,240
Trainable (router): 98,304  (0.0034%)



## Training data

The router needs to see both clinical and general text so it can learn the
contrast: Expert 1 reduces loss on clinical prompts; Expert 0 reduces it on
general ones. We use the same clinical dataset as chapter 7 plus a small
set of general-purpose sentences.

In [ ]:
dataset = load_dataset("oopere/clinical-ner-qdora", split="train")
train_dataset = dataset.select(range(300))

# Clinical prompts — same format the chapter-7 model was trained on.
clinical_prompts = []
for item in train_dataset:
    label = json.loads(item['label']) if isinstance(item['label'], str) else item['label']
    messages = [
        {"role": "system",    "content": "Extract:"},
        {"role": "user",      "content": item['note']},
        {"role": "assistant", "content": json.dumps(label, indent=2)}
    ]
    clinical_prompts.append(
        tokenizer.apply_chat_template(messages, tokenize=False,
                                      add_generation_prompt=False)
    )

from datasets import load_dataset

# Load SmolTalk — the same data SmolLM2 was instruction-tuned on
smoltalk = load_dataset("HuggingFaceTB/smoltalk", "all", split="train")

# Keep only single-turn conversations (one user + one assistant message)
# and short enough to be comparable to clinical prompts
smoltalk_single = smoltalk.filter(
    lambda x: len(x["messages"]) == 2 and                    # single turn only
              len(x["messages"][0]["content"]) < 300          # no huge prompts
)

general_subset = smoltalk_single.select(range(300))

# Build general_prompts directly from the messages column
general_prompts = []
for item in general_subset:
    general_prompts.append(
        tokenizer.apply_chat_template(
            item["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    )

training_prompts = clinical_prompts + general_prompts
random.shuffle(training_prompts)
print(f"Training prompts — clinical: {len(clinical_prompts)} | "
      f"general: {len(general_prompts)} | total: {len(training_prompts)}")

## Router training

The loop is structurally identical to Notebook 1. The difference is what
moves: here only the 6 router linear layers are updated — roughly 7,000
parameters in total across all MoE blocks.

The router discovers the right routing policy by itself. On a clinical
prompt Expert 1 produces lower loss; on a general prompt Expert 0 does.
The gradient flows back through the weighted sum and adjusts the router
weights accordingly. No routing labels are ever provided.

In [13]:
clear_gpu_cache()

In [ ]:
def train_MoE_router(model):
  model.config.use_cache = False
  model.to(DEVICE).train()

  # Promote only trainable parameters (router weights) to FP32 for stability.
  for param in model.parameters():
      if param.requires_grad:
          param.data = param.data.float()

  # Build the optimizer after dtype adjustment.
  router_params  = []

  for layer_idx in LAYERS_TO_MODIFY:
      router_params.extend(
          model.model.layers[layer_idx].mlp.router.parameters()
      )

  optimizer = optim.AdamW([
      {"params": router_params, "lr": 1e-3},
  ])

  scaler = GradScaler('cuda')

  for epoch in range(EPOCHS):
      total_loss = 0
      random.shuffle(training_prompts)

      for idx, text in enumerate(training_prompts):
          inputs = tokenizer(
              text, return_tensors="pt", truncation=True, max_length=512
          ).to(DEVICE)

          optimizer.zero_grad()

          with autocast('cuda', dtype=torch.float16):
              loss = model(**inputs, labels=inputs["input_ids"]).loss

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()

          total_loss += loss.item()
          if (idx + 1) % 100 == 0:
              print(f"  Epoch {epoch+1}  step {idx+1}/{len(training_prompts)}  loss: {loss.item():.4f}")

      print(f"-> END EPOCH {epoch+1:02d}  avg loss: {total_loss/len(training_prompts):.4f}\n")

  model.config.use_cache = True
  print("Router training complete.")

In [15]:
import gc
import torch.optim as optim
from torch.amp import autocast, GradScaler

gc.collect()
torch.cuda.empty_cache()


EPOCHS = 3
train_MoE_router(model)

  Epoch 1  step 100/600  loss: 0.7431
  Epoch 1  step 200/600  loss: 0.8998
  Epoch 1  step 300/600  loss: 1.8490
  Epoch 1  step 400/600  loss: 3.0067
  Epoch 1  step 500/600  loss: 1.4291
  Epoch 1  step 600/600  loss: 0.2879
-> END EPOCH 01  avg loss: 0.9070

  Epoch 2  step 100/600  loss: 1.1504
  Epoch 2  step 200/600  loss: 0.4724
  Epoch 2  step 300/600  loss: 0.6079
  Epoch 2  step 400/600  loss: 1.0791
  Epoch 2  step 500/600  loss: 0.5067
  Epoch 2  step 600/600  loss: 0.3044
-> END EPOCH 02  avg loss: 0.8960

  Epoch 3  step 100/600  loss: 1.9165
  Epoch 3  step 200/600  loss: 0.8555
  Epoch 3  step 300/600  loss: 1.6909
  Epoch 3  step 400/600  loss: 0.3971
  Epoch 3  step 500/600  loss: 1.0196
  Epoch 3  step 600/600  loss: 0.4151
-> END EPOCH 03  avg loss: 0.8943

Router training complete.


## Testing and evaluation

We evaluate whether the upcycled MoE keeps strong general behavior while improving clinical extraction, then inspect routing decisions and schema compliance.

### A/B test

We reuse the same prompts as Notebook 1. The expected result should be at least as good, because Expert 1 comes from a longer Chapter 7 specialization process.

In [16]:
model.to(torch.float16)
model.eval()


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Trainable2ExpertMoE(
          (expert_general): LlamaMLP(
            (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
            (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
            (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
            (act_fn): SiLUActivation()
          )
          (expert_clinical): LlamaMLP(
            (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)


In [17]:

def run_moe(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

print("=" * 52)
print("  A/B TEST RESULTS")
print("=" * 52)

# Test A — general knowledge
messages_general = [{"role": "user", "content": "What do you know about la Vall Fosca?"}]
prompt_general   = tokenizer.apply_chat_template(
    messages_general, tokenize=False, add_generation_prompt=True)
print("\n[Test A — General knowledge]")
print(f"Response: {run_moe(prompt_general, max_tokens=400)}")

# Test B — clinical NER
nota_test = ("Patient is a 50 yo F c/o severe chest pain and SOB since yesterday. "
             "Vitals: HR 110, BP 150/90. Started on Aspirin 81mg.")
messages_clinical = [
    {"role": "system", "content": "Extract:"},
    {"role": "user",   "content": nota_test}
]
prompt_clinical = tokenizer.apply_chat_template(
    messages_clinical, tokenize=False, add_generation_prompt=True)
print("\n[Test B — Clinical NER]")
print(f"Response: {run_moe(prompt_clinical, max_tokens=150)}")

  A/B TEST RESULTS

[Test A — General knowledge]
Response: La Vall Fosca is a beautiful and historic village located in the province of Gerona, in the northeastern part of Spain. It is situated in the Montseny Natural Park, which is a UNESCO Biosphere Reserve. The village is known for its stunning natural beauty, with its lush green hills, valleys, and forests.

The village has a rich history dating back to the Roman era, and it was an important center for the textile industry during the Middle Ages. Today, La Vall Fosca is a popular tourist destination, attracting visitors with its picturesque landscapes, traditional architecture, and rich cultural heritage.

The village is famous for its traditional Catalan cuisine, which includes dishes such as pa amb tomàquet (bread with tomato), escalivada (roasted vegetables), and xuixo (a sweet pastry). Visitors can also enjoy the beautiful views of the Montseny Natural Park, which offers hiking trails, waterfalls, and other outdoor activities.


## Routing analysis

After router training, we inspect token-level probabilities to verify that general prompts favor Expert 0 while clinical prompts increase Expert 1 activation.

In [18]:
def analyze_routing(text, layer_index=0):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _ = model(**inputs)
    moe_layer       = model.model.layers[layer_index].mlp
    routing_weights = moe_layer.last_routing.squeeze(0)
    tokens          = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    print(f"Routing analysis (layer {layer_index}):")
    print(f"{'Token':<18} | {'Expert 0 (General)':<20} | {'Expert 1 (Clinical)'}")
    print("-" * 65)
    for token, weights in zip(tokens, routing_weights):
        clean = token.replace("Ġ", "").replace(" ", "")
        print(f"{clean:<18} | {weights[0].item():<20.4f} | {weights[1].item():.4f}")

In [19]:
analyze_routing("Paris is the capital of France", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Paris              | 0.3347               | 0.6650
is                 | 0.5444               | 0.4556
the                | 0.7720               | 0.2279
capital            | 0.8438               | 0.1561
of                 | 0.8853               | 0.1147
France             | 0.9062               | 0.0935


In [20]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 0.2952               | 0.7051
is                 | 0.0012               | 0.9985
a                  | 0.0031               | 0.9971
                   | 0.0042               | 0.9956
6                  | 0.0199               | 0.9800
7                  | 0.0156               | 0.9844
-                  | 0.0168               | 0.9834
year               | 0.0225               | 0.9775
-                  | 0.0423               | 0.9575
old                | 0.0005               | 0.9995
male               | 0.0001               | 1.0000
with               | 0.0001               | 1.0000
chronic            | 0.0016               | 0.9985
pain               | 0.0012               | 0.9990


## Schema validation

We score generated responses against a strict JSON schema to measure extraction reliability by category and diagnose failure patterns.

In [ ]:
import json
import pandas as pd

REQUIRED_SCHEMA = {
    "patient_age": (int, type(None)),
    "symptoms": list,
    "vital_signs": dict,
    "medications": list,
    "duration_days": (int, type(None)),
}
VITAL_SIGNS_KEYS = {"temperature", "heart_rate", "blood_pressure"}

def check_schema_compliance(response_text: str) -> dict:
    # Step 1: remove possible markdown fences.
    cleaned = response_text.strip().removeprefix("```json").removesuffix("```").strip()

    # Step 2: parse JSON.
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {"is_valid_json": False, "matches_schema": False, "failure_reason": f"JSON parse error: {e}"}

    # Step 3: validate top-level keys and value types.
    for key, expected_type in REQUIRED_SCHEMA.items():
        if key not in parsed:
            return {"is_valid_json": True, "matches_schema": False, "failure_reason": f"Missing key: '{key}'"}
        if not isinstance(parsed[key], expected_type):
            return {"is_valid_json": True, "matches_schema": False, "failure_reason": f"Wrong type for '{key}': got {type(parsed[key]).__name__}"}

    # Step 4: validate required keys inside vital_signs.
    for vk in VITAL_SIGNS_KEYS:
        if vk not in parsed.get("vital_signs", {}):
            return {"is_valid_json": True, "matches_schema": False, "failure_reason": f"Missing vital_signs key: '{vk}'"}

    return {"is_valid_json": True, "matches_schema": True, "failure_reason": None}

### Evaluation loop on the clinical test split

We generate predictions for each test note, compute schema-compliance signals, and store per-sample results for aggregated reporting.

In [ ]:
def run_moe(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

def evaluate_moe_on_test_set():
    model.eval()  # Ensure no gradients are updated.
    results = []

    # Load the clinical dataset test split.
    test_dataset = load_dataset("oopere/clinical-ner-qdora", split="test")

    for sample in test_dataset:
        messages = [
            {"role": "system", "content": "Extract:"},
            {"role": "user",   "content": sample['note']}
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

        response = run_moe(prompt, max_tokens=400)
        compliance = check_schema_compliance(response)

        results.append({
            "category": sample["category"],
            "model": "MoE_Upcycled",
            "is_valid_json": compliance["is_valid_json"],
            "matches_schema": compliance["matches_schema"],
            "failure_reason": compliance["failure_reason"],
        })

    return pd.DataFrame(results)

In [23]:
def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    summary = df.groupby("category").agg(
        samples=("is_valid_json", "count"),
        valid_json=("is_valid_json", "sum"),
        schema_ok=("matches_schema", "sum")
    )
    summary["valid_json_pct"] = (summary["valid_json"] / summary["samples"] * 100).round(1)
    summary["schema_ok_pct"] = (summary["schema_ok"] / summary["samples"] * 100).round(1)
    return summary

print("Evaluating MoE upcycled model...")
moe_results = evaluate_moe_on_test_set()

print("\nMoE Schema compliance by category:")
print(summarize_results(moe_results)[["samples", "valid_json_pct", "schema_ok_pct"]].to_string())

moe_score = moe_results["matches_schema"].mean() * 100
print(f"\nMoE overall schema compliance: {moe_score:.1f}%")

Evaluating MoE upcycled model...

MoE Schema compliance by category:
               samples  valid_json_pct  schema_ok_pct
category                                             
abbreviations        6           100.0          100.0
clean                8           100.0          100.0
implicit             7           100.0          100.0
irrelevant          12            58.3           58.3
typos                7           100.0          100.0

MoE overall schema compliance: 87.5%


### Failure-case inspection

After aggregate scores, we inspect failing samples in the irrelevant category to understand concrete schema mismatch patterns.

In [24]:
# Inspect failing cases in the irrelevant category
test_dataset = load_dataset("oopere/clinical-ner-qdora", split="test")
irrelevant_samples = [s for s in test_dataset if s["category"] == "irrelevant"]

print(f"Irrelevant samples: {len(irrelevant_samples)}\n")
print("=" * 70)

for i, sample in enumerate(irrelevant_samples):
    messages = [
        {"role": "system", "content": "Extract:"},
        {"role": "user",   "content": sample['note']}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = run_moe(prompt, max_tokens=400)
    compliance = check_schema_compliance(response)

    # Only print failing cases
    if not compliance["matches_schema"]:
        print(f"\n--- FAILING CASE {i+1} ---")
        print(f"NOTE:\n{sample['note']}")
        print(f"\nRESPONSE:\n{response}")
        print(f"\nFAILURE REASON: {compliance['failure_reason']}")
        print("=" * 70)

Irrelevant samples: 12


--- FAILING CASE 2 ---
NOTE:
Pt scheduled for 2:30 PM but arrived 45 mins late due to traffic. Insurance pre-auth pending for specialist referral. 67-year-old retired teacher presents with worsening joint stiffness and pain in hands and knees over past 3 weeks. Patient's daughter accompanied her today - very concerned about mother's mobility. Vital signs: T 99.1F, HR 88, BP not obtained due to equipment malfunction. Started on ibuprofen 400mg TID last week per phone consultation. Patient mentions she's been stressed about upcoming move to assisted living facility. Previous visit 6 months ago was for routine physical - all normal at that time. Dr. Martinez reviewed labs from 2 weeks ago - ESR elevated at 45. Patient reports morning stiffness lasting about 2 hours, difficulty opening jars. Denies fever, weight loss, or rash. Social history: lives alone, no smoking, occasional wine with dinner. Need to follow up in 2 weeks. Nurse noted patient seemed anxious about

## Fine-tuning the clinical expert tail

As a final step, we unfreeze the last clinical-expert layers to adapt them to the upcycled MoE context and reassess routing and schema compliance.

In [25]:
# ============================================================
# Unfreezing the last 6 layers of Expert 1 for fine-tuning
# ============================================================
# The clinical expert was trained with QLoRA-modified attention layers.
# Its MLP now receives hidden states from a different attention distribution.
# Allowing the last layers to adapt closes that gap.

EXPERT_LAYERS_TO_UNFREEZE = range(18, 24)
for layer_idx in EXPERT_LAYERS_TO_UNFREEZE:
    for param in model.model.layers[layer_idx].mlp.expert_clinical.parameters():
        param.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable (router + last 6 expert layers): {trainable:,}  ({trainable/total:.4%})")

Trainable (router + last 6 expert layers): 302,088,192  (10.3475%)


In [ ]:
def train_MoE_router_expert(model):
  model.config.use_cache = False
  model.to(DEVICE).train()

  # Promote only trainable parameters (router + selected expert layers) to FP32.
  for param in model.parameters():
      if param.requires_grad:
          param.data = param.data.float()

  # Build optimizer parameter groups after dtype adjustment.
  router_params  = []
  expert_params  = []

  for layer_idx in EXPERT_LAYERS_TO_UNFREEZE:
      router_params.extend(
          model.model.layers[layer_idx].mlp.router.parameters()
      )
      expert_params.extend(
          model.model.layers[layer_idx].mlp.expert_clinical.parameters()
      )

  optimizer = optim.AdamW([
      {"params": router_params,  "lr": 1e-3},
      {"params": expert_params,  "lr": 1e-5},
  ])

  scaler = GradScaler('cuda')

  for epoch in range(EPOCHS):
      total_loss = 0
      random.shuffle(training_prompts)

      for idx, text in enumerate(training_prompts):
          inputs = tokenizer(
              text, return_tensors="pt", truncation=True, max_length=512
          ).to(DEVICE)

          optimizer.zero_grad()

          with autocast('cuda', dtype=torch.float16):
              loss = model(**inputs, labels=inputs["input_ids"]).loss

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()

          total_loss += loss.item()
          if (idx + 1) % 100 == 0:
              print(f"  Epoch {epoch+1}  step {idx+1}/{len(training_prompts)}  loss: {loss.item():.4f}")

      print(f"-> END EPOCH {epoch+1:02d}  avg loss: {total_loss/len(training_prompts):.4f}\n")

  model.config.use_cache = True
  print("Router training complete.")

In [27]:
EPOCHS=1
train_MoE_router_expert(model)

  Epoch 1  step 100/600  loss: 0.1554
  Epoch 1  step 200/600  loss: 0.2670
  Epoch 1  step 300/600  loss: 0.4200
  Epoch 1  step 400/600  loss: 0.2442
  Epoch 1  step 500/600  loss: 0.2965
  Epoch 1  step 600/600  loss: 0.5284
-> END EPOCH 01  avg loss: 0.5556

Router training complete.


In [28]:
analyze_routing("Paris is the capital of France", layer_index=18)
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=18)

Routing analysis (layer 18):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Paris              | 0.4512               | 0.5488
is                 | 0.0141               | 0.9858
the                | 0.1022               | 0.8979
capital            | 0.0480               | 0.9521
of                 | 0.0469               | 0.9531
France             | 0.0114               | 0.9888
Routing analysis (layer 18):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 0.4512               | 0.5488
is                 | 0.0000               | 1.0000
a                  | 0.0000               | 1.0000
                   | 0.0000               | 1.0000
6                  | 0.0000               | 1.0000
7                  | 0.0001               | 1.0000
-                  | 0.0000               | 1.0000
year               

In [29]:
analyze_routing("Paris is the capital of France", layer_index=23)
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Paris              | 0.1537               | 0.8462
is                 | 0.3413               | 0.6587
the                | 0.5781               | 0.4216
capital            | 0.7544               | 0.2457
of                 | 0.6938               | 0.3059
France             | 0.8833               | 0.1166
Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 0.1974               | 0.8027
is                 | 0.0002               | 1.0000
a                  | 0.0002               | 0.9995
                   | 0.0009               | 0.9990
6                  | 0.0067               | 0.9932
7                  | 0.0044               | 0.9956
-                  | 0.0074               | 0.9927
year               

In [30]:
model.to(torch.float16)
model.eval()
print("Evaluating MoE upcycled model...")

moe_results = evaluate_moe_on_test_set()

print("\nMoE Schema compliance by category:")
print(summarize_results(moe_results)[["samples", "valid_json_pct", "schema_ok_pct"]].to_string())

moe_score = moe_results["matches_schema"].mean() * 100
print(f"\nMoE overall schema compliance: {moe_score:.1f}%")

Evaluating MoE upcycled model...

MoE Schema compliance by category:
               samples  valid_json_pct  schema_ok_pct
category                                             
abbreviations        6           100.0          100.0
clean                8           100.0          100.0
implicit             7           100.0          100.0
irrelevant          12           100.0          100.0
typos                7           100.0          100.0

MoE overall schema compliance: 100.0%


## What changed between the two notebooks

|  | Notebook 1 | Notebook 2 |
|---|---|---|
| Expert 0 | base model MLP | base model MLP |
| Expert 1 | copy of base MLP, trained from scratch | MLP from chapter 7 model |
| Phase 1 (Expert 1 training) | 300 examples × 3 epochs | not needed |
| Phase 2 (Router training) | mixed data | mixed data |
| Trainable parameters | ~24% (expert + router) | ~0.003% → ~10% |

The chapter-7 model was not built for this chapter. It was built to solve a
specific problem — clinical entity extraction — and it did. The fact that its
MLP weights can be lifted out and reused as an expert inside a completely
different architecture is not a coincidence: it is a direct consequence of
how the Transformer separates concerns. Attention is shared; feed-forward
knowledge is local and transferable.

That transferability is what Sparse Upcycling formalises.
